In [3]:
import pandas as pd

df_train = pd.read_csv("validation/validation_data.csv")
df_test = pd.read_csv("test/test_pairs.csv")

df_train.head()

,tile1,tile2,answer
0,985497,820187,0
1,504151,527996,5
2,411829,464134,0
3,820187,893118,1
4,251646,471515,0


In [19]:
from torchvision import models
from PIL import Image
import numpy as np
from torch import nn
import torch
import tqdm
import torchvision.transforms as T
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = nn.Identity()
resnet = resnet.to("cuda")
X = []

transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor()
])

for _, row in tqdm.tqdm(df_train.iterrows()):
    img1 = transform(Image.open(f'validation/validation_tiles/{row.tile1}.jpg')).unsqueeze(0).to("cuda")
    X1 = resnet(img1).squeeze(0).cpu().detach().numpy()
    img2 = transform(Image.open(f'validation/validation_tiles/{row.tile2}.jpg')).unsqueeze(0).to("cuda")
    X2 = resnet(img2).squeeze(0).cpu().detach().numpy()
    full = np.hstack([X1, X2])
    X.append(full)

2470it [00:47, 52.28it/s]


In [21]:
X = np.array(X)
y = df_train.answer

X.shape

(2470, 1024)

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=3, scoring='f1_macro').mean()

np.float64(0.11247150133413653)

In [28]:
model.fit(X, y)

LinearSVC()

In [30]:
X_final = []

for _, row in tqdm.tqdm(df_test.iterrows()):
    img1 = transform(Image.open(f'test/test_tiles/{row.tile1}.jpg')).unsqueeze(0).to("cuda")
    X1 = resnet(img1).squeeze(0).cpu().detach().numpy()
    img2 = transform(Image.open(f'test/test_tiles/{row.tile2}.jpg')).unsqueeze(0).to("cuda")
    X2 = resnet(img2).squeeze(0).cpu().detach().numpy()
    full = np.hstack([X1, X2])
    X_final.append(full)

5360it [01:50, 48.40it/s]


In [31]:
X_final = np.array(X_final)

X_final.shape

(5360, 1024)

In [43]:
y_preds = model.predict(X_final)

In [48]:
df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.datapointID,
    "answer": y_preds
})

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.datapointID,
    "answer": y_preds
})

In [49]:
submission = pd.concat([df_1, df_2])
submission

,subtaskID,datapointID,answer
0,1,1,0
1,1,2,0
2,1,3,5
3,1,4,5
4,1,5,0
...,...,...,...
5355,2,5356,0
5356,2,5357,5
5357,2,5358,5
5358,2,5359,5


In [50]:
submission.answer.value_counts()

answer
0    7194
5    3374
2      46
4      44
3      32
1      30
Name: count, dtype: int64

In [51]:
submission.to_csv("submission.csv", index=False)